# NPPES — Delaware peek

Scope: Delaware primary-care providers (identity, practice location, taxonomy).
Source: `data/sources/provider_registry/npi_files/NPPES_Data_Dissemination_September_2026_V2.zip`
Reading: chunked pandas, filtered to practice-location state = DE. Main file is ~11.7 GB uncompressed.

In [2]:
import zipfile
from pathlib import Path

import pandas as pd

# Resolve project root (this notebook lives in notebooks/).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ZIP = ROOT / "data/sources/provider_registry/npi_files/NPPES_Data_Dissemination_September_2026_V2.zip"
MEMBER = "npidata_pfile_20050523-20260913.csv"
STATE_COL = "Provider Business Practice Location Address State Name"
STATE = "DE"
print("zip exists:", ZIP.exists())

zip exists: True


In [3]:
# Peek: columns + first rows, no full load.
with zipfile.ZipFile(ZIP) as z, z.open(MEMBER) as f:
    sample = pd.read_csv(f, nrows=5, dtype=str)
print(sample.shape[1], "columns")
sample.head()

330 columns


,NPI,Entity Type Code,Replacement NPI,Employer Identification Number (EIN),Provider Organization Name (Legal Business Name),Provider Last Name (Legal Name),Provider First Name,Provider Middle Name,Provider Name Prefix Text,Provider Name Suffix Text,...,Healthcare Provider Taxonomy Group_7,Healthcare Provider Taxonomy Group_8,Healthcare Provider Taxonomy Group_9,Healthcare Provider Taxonomy Group_10,Healthcare Provider Taxonomy Group_11,Healthcare Provider Taxonomy Group_12,Healthcare Provider Taxonomy Group_13,Healthcare Provider Taxonomy Group_14,Healthcare Provider Taxonomy Group_15,Certification Date
0,1679576722,1,NaN,NaN,NaN,WIEBE,DAVID,A,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1588667638,1,NaN,NaN,NaN,PILCHER,WILLIAM,C,DR.,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1497758544,2,NaN,<UNAVAIL>,"CUMBERLAND COUNTY HOSPITAL SYSTEM, INC",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1306849450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1215930367,1,NaN,NaN,NaN,GRESSOT,LAURENT,NaN,DR.,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Scope to Delaware by practice-location state. Chunked scan (heavy: reads the full file once).
TAX = [f"Healthcare Provider Taxonomy Code_{i}" for i in range(1, 16)]
SWITCH = [f"Healthcare Provider Primary Taxonomy Switch_{i}" for i in range(1, 16)]
KEEP = [
    "NPI", "Entity Type Code",
    "Provider Organization Name (Legal Business Name)",
    "Provider Last Name (Legal Name)", "Provider First Name",
    "Provider First Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    STATE_COL,
    "Provider Business Practice Location Address Postal Code",
    "NPI Deactivation Date",
] + TAX + SWITCH

parts = []
with zipfile.ZipFile(ZIP) as z, z.open(MEMBER) as f:
    for chunk in pd.read_csv(f, usecols=KEEP, dtype=str, chunksize=200_000):
        parts.append(chunk[chunk[STATE_COL] == STATE])
de = pd.concat(parts, ignore_index=True)
de.shape

(27975, 40)

In [5]:
# Entity split (1=individual, 2=org) and primary-taxonomy distribution.
print(de["Entity Type Code"].value_counts(dropna=False).to_string())

def primary_taxonomy(row):
    for i in range(1, 16):
        if row[f"Healthcare Provider Primary Taxonomy Switch_{i}"] == "Y":
            return row[f"Healthcare Provider Taxonomy Code_{i}"]
    return None

de["primary_taxonomy"] = de.apply(primary_taxonomy, axis=1)
de["primary_taxonomy"].value_counts().head(20)

Entity Type Code
1    21634
2     6341


primary_taxonomy
225100000X    1252
1041C0700X    1156
163W00000X     978
101YM0800X     961
183500000X     869
207Q00000X     775
363LF0000X     766
390200000X     760
207R00000X     755
363A00000X     697
235Z00000X     694
163WS0200X     601
363L00000X     600
101YA0400X     555
225X00000X     518
111N00000X     447
103TS0200X     422
363LP0808X     376
208000000X     371
1223G0001X     360
Name: count, dtype: int64

In [16]:
de.keys()

Index(['NPI', 'Entity Type Code',
       'Provider Organization Name (Legal Business Name)',
       'Provider Last Name (Legal Name)', 'Provider First Name',
       'Provider First Line Business Practice Location Address',
       'Provider Business Practice Location Address City Name',
       'Provider Business Practice Location Address State Name',
       'Provider Business Practice Location Address Postal Code',
       'NPI Deactivation Date', 'Healthcare Provider Taxonomy Code_1',
       'Healthcare Provider Primary Taxonomy Switch_1',
       'Healthcare Provider Taxonomy Code_2',
       'Healthcare Provider Primary Taxonomy Switch_2',
       'Healthcare Provider Taxonomy Code_3',
       'Healthcare Provider Primary Taxonomy Switch_3',
       'Healthcare Provider Taxonomy Code_4',
       'Healthcare Provider Primary Taxonomy Switch_4',
       'Healthcare Provider Taxonomy Code_5',
       'Healthcare Provider Primary Taxonomy Switch_5',
       'Healthcare Provider Taxonomy Code_6',
 

In [21]:
cols = [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]

df_filtered = de[de[cols].notna().all(axis=1)]

df_filtered.head()

,NPI,Entity Type Code,Provider Organization Name (Legal Business Name),Provider Last Name (Legal Name),Provider First Name,Provider First Line Business Practice Location Address,Provider Business Practice Location Address City Name,Provider Business Practice Location Address State Name,Provider Business Practice Location Address Postal Code,NPI Deactivation Date,...,Healthcare Provider Primary Taxonomy Switch_11,Healthcare Provider Taxonomy Code_12,Healthcare Provider Primary Taxonomy Switch_12,Healthcare Provider Taxonomy Code_13,Healthcare Provider Primary Taxonomy Switch_13,Healthcare Provider Taxonomy Code_14,Healthcare Provider Primary Taxonomy Switch_14,Healthcare Provider Taxonomy Code_15,Healthcare Provider Primary Taxonomy Switch_15,primary_taxonomy
899,1346220266,2,"FIRST STATE ORTHOPAEDICS, PA",NaN,NaN,4745 OGLETOWN STANTON RD STE 225,NEWARK,DE,197131340,NaN,...,N,208100000X,N,208VP0014X,N,213E00000X,N,225100000X,N,207X00000X
1652,1730133992,2,BEEBE PHYSICIAN NETWORK INC.,NaN,NaN,1515 SAVANNAH RD,LEWES,DE,199581675,NaN,...,N,208M00000X,N,363A00000X,N,363L00000X,N,363LN0000X,N,207R00000X
1995,1639113285,1,NaN,KHAN,MOHAMMAD,2600 GLASGOW AVE STE 212,NEWARK,DE,197025704,NaN,...,N,2080P0207X,N,207PP0204X,N,207QA0401X,N,208000000X,N,208000000X
4811,1457400657,2,NORMAN BROUDY,NaN,NaN,825 N WASHINGTON ST,WILMINGTON,DE,198011509,NaN,...,N,104100000X,N,104100000X,N,104100000X,N,2084P0800X,Y,2084P0800X
4973,1366594699,2,THE NEMOURS FOUNDATION,NaN,NaN,1600 ROCKLAND RD,WILMINGTON,DE,198033607,NaN,...,Y,261QM2500X,N,261QP2300X,N,261QR0200X,N,261QX0200X,N,261QM1300X


## Notes
- Scope: NPPES providers whose **practice-location** state = DE (not mailing).
- Next: map `primary_taxonomy` to the primary-care scope list (HISTORY 001) via NUCC codes.
- Watch: deactivated NPIs (`NPI Deactivation Date` present); org rows (Entity Type 2) have blank person-name fields.